## https://github.com/agungperdananto/2026_class

Berikut adalah simulasi setiap fasenya dengan contoh input dan outputnya berdasarkan sumber
materi:
### 1. Analisis Leksikal (Lexical Analyzer/Tokenizer)
Fase ini memecah aliran karakter dari kode sumber menjadi unit bermakna yang disebut token
Python dapat menggunakan modul re (ekspresi regular) atau ply.lex untuk mengenali pola
token .
●
Input: String kode sumber, contoh: y = a + 2 * b .
●
Output: Aliran token (sering kali direpresentasikan sebagai tuple tipe dan nilai),
contoh:('ID', 'y'), ('ASSIGNMENT','='), ('ID','a'), 
('PLUS', '+'), ('NUMBER', 2), ('TIMES','*'),
('ID', 'b') .
### 2. Analisis Sintaksis (Syntax Analyzer/Parser)
Fase ini memeriksa apakah urutan token sesuai dengan aturan tata bahasa (grammar) yang
telah ditentukan (misalnya menggunakan format EBNF) . Output utamanya biasanya berupa
pohon struktur data 10.
Input: Aliran token hasil analisis leksikal 11.
Output: Abstract Syntax Tree (AST) atau Parse Tree .
●
Contoh visual AST untuk 2 * 7 + 3: Sebuah pohon biner dengan node akar +
, anak kiri
berupa sub-pohon 2 * 7, dan anak kanan berupa angka .

### 3. Analisis Semantik
Fase ini memvalidasi kebenaran logika program di luar struktur tata bahasanya, seperti
memeriksa apakah suatu variabel sudah didefinisikan sebelum digunakan atau pengecekan tipe
data 4, 16. Di Python, ini sering diimplementasikan menggunakan Symbol Table (tabel simbol)
berbasis dictionary.
●
Input: AST dan Environment (kamus variabel), contoh: expression: 9 * x dan env: {'x':
11} .
●
Output: Nilai hasil evaluasi atau pesan kesalahan jika ada pelanggaran logika.
●
Contoh Output: 99 (jika benar) atau ParserError: Undefined variable 'x' (jika variabel
tidak ada di tabel simbol).
### 4. Generasi Kode Antara (Intermediate Code Generation)
Fase ini menghasilkan representasi program yang lebih sederhana dan independen dari mesin
target, biasanya dalam bentuk Three Address Code (TAC) . Dalam TAC, setiap instruksi
memiliki maksimal tiga operan .
1. Input: AST yang telah divalidasi secara semantik.
2. Output: Rangkaian instruksi linier, contoh untuk ekspresi x + y * z :
3. t1 = y * z
4. t2 = x + t1(Di mana t1 dan t2 adalah nama sementara yang dihasilkan oleh kompilator)

In [1]:
import re
# --- PHASE 0: ERROR HANDLING ---
class ParserError(Exception):
    """Kelas khusus untuk menangani kesalahan saat proses parsing [5]."""
    pass

# --- PHASE 2: AST NODES (REPRESENTASI POHON) ---
class AST:
    pass

class BinOp(AST):
    """Node untuk operasi biner seperti + atau * [2]."""
    def __init__(self, left, op, right):
        self.left = left
        self.op = op
        self.right = right

class Num(AST):
    """Node untuk menyimpan angka [6]."""
    def __init__(self, value):
        self.value = value

class Var(AST):
    """Node untuk menyimpan nama variabel [7]."""
    def __init__(self, name):
        self.name = name


In [2]:
# --- PHASE 1 & 2: LEXER & PARSER ---
class MiniCompiler:
    def __init__(self, source, env):
        # Analisis Leksikal menggunakan Regular Expressions [8]
        self._tokens = iter(re.findall(r'[a-zA-Z_]\w*|\d+(?:\.\d+)?|[+*/()-]', source) + ['?'])
        self._current = None
        self._env = env  # Tabel Simbol (Semantic) [9]
        self._temp_count = 0 # Untuk TAC [4]
        self.advance()

    def advance(self):
        """Mengambil token berikutnya dari aliran input [10]."""
        try:
            self._current = next(self._tokens)
        except StopIteration:
            self._current = None

    def expect(self, expected):
        """Memvalidasi apakah token saat ini sesuai dengan gramatika [11]."""
        if self._current != expected and not (expected == "ID" and self._current.isalnum()):
            raise ParserError(f"Expected {expected}, found {self._current}")
        token = self._current
        self.advance()
        return token

    # Analisis Sintaksis (Grammar Rules) [7, 12]
    def factor(self):
        token = self._current
        # print(f"{token=}")
        if token is not None and token.replace('.', '', 1).isdigit():
            # print(f"angka:{token.replace('.', '', 1).isdigit()}")
            self.advance()
            return Num(float(token) if '.' in token else int(token))
        elif token.isalpha():
            # print(f"huruf:{token.isalpha()}")
            # Analisis Semantik: Cek apakah variabel terdefinisi [13]
            if token not in self._env:

                raise ParserError(f"Semantic Error: Undefined variable '{token}', symbol table {self._env}")
            self.advance()
            return Var(token)
        elif token == '(':
            self.advance()
            node = self.expr()
            self.expect(')')
            return node

    # perkalian dan pembagian
    def term(self):
        node = self.factor()
        while self._current in ('*', '/'):
            op = self._current
            self.advance()
            node = BinOp(left=node, op=op, right=self.factor())
        return node

    # penjumlahan dan pengurangan
    def expr(self):
        node = self.term()
        while self._current in ('+', '-'):
            op = self._current
            self.advance()
            node = BinOp(left=node, op=op, right=self.term())
        return node

    # buat function untuk akar(//) dan pangkat(^) seperti term() dan expr()
    def root(self):
        pass

    # --- PHASE 4: GENERASI KODE ANTARA (TAC) ---
    def generate_tac(self, node):
        """Menghasilkan Three Address Code secara rekursif [4, 14]."""
        if isinstance(node, Num):
            return str(node.value)
        if isinstance(node, Var):
            return node.name
        
        left_val = self.generate_tac(node.left)
        right_val = self.generate_tac(node.right)
        
        self._temp_count += 1
        temp_name = f"t{self._temp_count}"
        print(f"{temp_name} = {left_val} {node.op} {right_val}") # Output TAC [15]
        return temp_name

In [6]:
# --- SIMULASI EKSEKUSI ---
source_code = "a + b * (5 - z)"
symbol_table = {'a': 10, 'b': 5, 'c': 10, 'z': 11.7} # Variabel terdefinisi (Analisis Semantik) [9]
print(f"Input Kode Sumber: {source_code}")
print("-" * 30)

try:
    compiler = MiniCompiler(source_code, symbol_table)
    ast_root = compiler.expr() # Mulai proses parsing
    
    print("Output Analisis Semantik: Valid (Semua variabel ditemukan)")
    print("-" * 30)
    print("Output Generasi Kode Antara (Three Address Code):")
    compiler.generate_tac(ast_root)
    
except ParserError as e:
    print(e)

Input Kode Sumber: a + b * (5 - z)
------------------------------
Output Analisis Semantik: Valid (Semua variabel ditemukan)
------------------------------
Output Generasi Kode Antara (Three Address Code):
t1 = 5 - z
t2 = b * t1
t3 = a + t2


Penjelasan Output Simulasi:
1. Analisis Leksikal: Input string a + 2 * b dipecah oleh regex menjadi unit token: ['a', '+', '2', '*', 'b'].
2. Analisis Sintaksis: Parser membangun Abstract Syntax Tree (AST). Node perkalian (2 * b) diletakkan lebih rendah dari penjumlahan karena memiliki prioritas lebih tinggi.
3. Analisis Semantik: Saat parser menemukan variabel a dan b, ia memeriksa symbol_table (Environment). Jika variabel tidak ada di kamus tersebut, compiler akan melempar ParserError.
4. Generasi Kode Antara (TAC): Pohon AST ditelusuri menggunakan metode postorder traversal untuk menghasilkan instruksi linier dengan maksimal tiga alamat